# 11 CHECK_BLOWER: results by fault group

In June 2025, 367 of the 593 onsets (61.9 per cent) carry the message `AUX CAP MACH RM BLOW1 NOT OFF`, a supervision artefact (Section 2.2 of the paper). This notebook answers three questions:

1. **Debounce.** Are the windows built from all onsets?
2. **Operating mode.** Are blower onsets logged at standstill, or after a change of operating state, more often than other onsets?
3. **Performance.** Does the primary XGBoost result hold when blower onsets are removed from the positives? It is also run on the blower onsets alone.

Inputs: `build/fault_episodes_v4.csv`, `build/telemetry_labelled_v4.pkl`; outputs in `OUTPUT/v4/` with the suffix `_check`.

Paper: Section 3.2; Table 10.

In [ ]:
import numpy as np, pandas as pd, warnings
from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

CFG = dict(
    labelled   = "build/telemetry_labelled_v4.pkl",
    episodes   = "build/fault_episodes_v4.csv",
    outdir     = "OUTPUT/v4",
    blower     = "AUX CAP MACH RM BLOW1 NOT OFF",
    horizons   = ["D5", "W5", "W60", "W300"],
    windows    = {"D5": (-5, 5), "W5": (-5, -1), "W60": (-60, -1), "W300": (-300, -1)},
    patience   = 100, test_frac = 0.20, val_frac = 0.20, seed = 42,
    sample_frac = 1.0,     # < 1.0 only for a fast trial run
)
NUMERIC = ["KMH", "DIST_M", "KV", "AMP", "KWH", "BV", "PRESSURE", "TRACTION_PCT", "BG1_KN", "BG2_KN", "BG1_DCLV", "BG2_DCLV"]
WHEEL_SLIP = ["Wheel_Slip_TM2", "Wheel_Slip_TM3", "Wheel_Slip_TM4", "Wheel_Slip_TM5", "Wheel_Slip_TM6"]
PRIMARY = NUMERIC + WHEEL_SLIP                      # the 17 channels of the primary specification
TRACTION_MODES = ["POWERING", "PROPULSION", "REGENERATIVE BRAKING", "VCB ON"]

tel = pd.read_pickle(CFG["labelled"]).sort_values("DATETIME").reset_index(drop=True)
if CFG["sample_frac"] < 1.0:
    tel = tel.iloc[:int(len(tel) * CFG["sample_frac"])].copy()
t0, t1 = tel["DATETIME"].min(), tel["DATETIME"].max()
ep = pd.read_csv(CFG["episodes"], parse_dates=["onset", "recovery"])
ep = ep[(ep.onset >= t0) & (ep.onset <= t1)].copy()
ep["is_blower"] = ep["fault"].eq(CFG["blower"])
print(f"{len(tel):,} telemetry rows, {t0} to {t1}")
print(f"{len(ep)} onsets in that period, {ep.is_blower.sum()} blower ({100*ep.is_blower.mean():.1f}%)")

## 1. Debounce: which onsets are used?

In [ ]:
ts = tel["DATETIME"].values.astype("datetime64[s]")
pos = pd.Series(np.arange(len(tel)), index=pd.DatetimeIndex(ts))

def label_before(onsets, h, lag=1):
    """Label of horizon h at the second `lag` s before each onset (NaN if no telemetry)."""
    idx = pos.reindex(pd.DatetimeIndex(onsets) - pd.Timedelta(seconds=lag))
    return tel[h].values[idx.dropna().astype(int)], idx.notna().values

rows = []
for keep in [True, False]:
    sub = ep[ep.debounce_keep == keep]
    lab, has = label_before(sub.onset, "W5")
    rows.append(dict(debounce_keep=keep, onsets=len(sub), with_telemetry=int(has.sum()),
                     W5_positive_pct=round(100 * np.mean(lab == 1), 1),
                     W5_excluded_pct=round(100 * np.mean(lab == -1), 1)))
deb = pd.DataFrame(rows).set_index("debounce_keep")
print(deb.to_string())
USE_ALL = deb.loc[False, "W5_positive_pct"] > 0.5 * deb.loc[True, "W5_positive_pct"]
used = ep if USE_ALL else ep[ep.debounce_keep]
print(f"\n-> the labels use {'ALL onsets' if USE_ALL else 'only debounce_keep = True onsets'}; "
      f"{len(used)} onsets used below ({used.is_blower.sum()} blower).")
deb.to_csv(f"{CFG['outdir']}/debounce_check.csv")

## 2. Operating mode at onset: blower versus other messages

Mode at the onset second and 10 s earlier, whether it changed in between, and whether the locomotive was stationary at onset. Grouped as in Table 5 (idle and transitional states pooled).

In [ ]:
def at(onsets, lag):
    idx = pos.reindex(pd.DatetimeIndex(onsets) - pd.Timedelta(seconds=lag))
    return idx

def mode_group(op):
    op = op.astype(str)
    return op.where(op.isin(TRACTION_MODES), "IDLE / TRANSITIONAL")

i0, i10 = at(used.onset, 0), at(used.onset, 10)
ok = i0.notna().values & i10.notna().values
u = used[ok].copy()
u["mode_at_onset"]  = mode_group(tel["OP"].iloc[i0[ok].astype(int)]).values
u["mode_10s_before"] = mode_group(tel["OP"].iloc[i10[ok].astype(int)]).values
u["mode_changed"]   = u.mode_at_onset != u.mode_10s_before
u["stationary"]     = tel["KMH"].iloc[i0[ok].astype(int)].values == 0
u["group"] = np.where(u.is_blower, "Blower label", "All other labels")

modes = pd.crosstab(u.group, u.mode_at_onset, normalize="index").mul(100).round(1)
extra = u.groupby("group").agg(onsets=("fault", "size"),
                               mode_changed_pct=("mode_changed", lambda s: round(100 * s.mean(), 1)),
                               stationary_pct=("stationary", lambda s: round(100 * s.mean(), 1)))
mode_tab = extra.join(modes)
print(f"{ok.sum()} of {len(used)} onsets have telemetry at onset and 10 s before\n")
print(mode_tab.T.to_string())
mode_tab.to_csv(f"{CFG['outdir']}/mode_at_onset_check.csv")

## 3. Primary XGBoost with and without the blower onsets

Same protocol as `07_CLASSIFY`: 17 primary channels, default class weighting, chronological three-way split, early stopping on validation, refit on the full training partition, test partition scored once.

* **All onsets** reproduces the primary row of Table 9.
* **Without blower**: seconds that are positive only because of a blower onset are set to excluded (-1), so they count neither as warnings nor as ordinary running.
* **Blower only**: the same, the other way round.

In [ ]:
def chrono_cut(y, frac):
    n = len(y); i = int(n * (1 - frac))
    while i < n - 1 and y[i] == 1 and y[i - 1] == 1:
        i += 1
    return i

def three_way_split(y):
    n = len(y); t = int(n * (1 - CFG["test_frac"]))
    while t < n - 1 and y[t] == 1:
        t += 1
    v = chrono_cut(y[:t], CFG["val_frac"])
    return np.arange(v), np.arange(v, t), np.arange(t), np.arange(t, n)

def xgb(y, n_iter=None):
    spw = float((y == 0).sum() / max((y == 1).sum(), 1))
    kw = dict(n_estimators=600, early_stopping_rounds=CFG["patience"]) if n_iter is None else dict(n_estimators=n_iter)
    return XGBClassifier(learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                         eval_metric="aucpr", tree_method="hist", scale_pos_weight=spw,
                         random_state=CFG["seed"], n_jobs=-1, **kw)

def fit_score(X, y):
    ia, iv, itr, ite = three_way_split(y)
    m1 = xgb(y[ia]).fit(X[ia], y[ia], eval_set=[(X[iv], y[iv])], verbose=False)
    n_iter = int(m1.best_iteration) + 1
    m2 = xgb(y[itr], n_iter).fit(X[itr], y[itr])
    s = m2.predict_proba(X[ite])[:, 1]
    base = y[ite].mean()
    ap = average_precision_score(y[ite], s)
    return dict(pr_auc=round(ap, 3), baseline=round(base, 4), lift=round(ap / base, 1),
                test_positive_s=int(y[ite].sum()), trees=n_iter)

def coverage(onsets, lo, hi):
    """Number of onset windows [onset+lo, onset+hi] covering each telemetry row."""
    d = np.zeros(len(ts) + 1, dtype=np.int32)
    o = pd.DatetimeIndex(onsets).values.astype("datetime64[s]")
    a = np.searchsorted(ts, o + np.timedelta64(lo, "s"), side="left")
    b = np.searchsorted(ts, o + np.timedelta64(hi, "s"), side="right")
    np.add.at(d, a, 1); np.add.at(d, b, -1)
    return np.cumsum(d)[:-1]

X_all = tel[PRIMARY].astype("float32").fillna(0).values
res = []
for h in tqdm(CFG["horizons"], desc="horizons"):
    lo, hi = CFG["windows"][h]
    cov_b = coverage(used.onset[used.is_blower], lo, hi)
    cov_o = coverage(used.onset[~used.is_blower], lo, hi)
    y0 = tel[h].astype(int).values
    variants = {
        "All onsets (primary)": y0,
        "Without blower":       np.where((y0 == 1) & (cov_b > 0) & (cov_o == 0), -1, y0),
        "Blower only":          np.where((y0 == 1) & (cov_o > 0) & (cov_b == 0), -1, y0),
    }
    for name, y in variants.items():
        keep = y != -1
        r = fit_score(X_all[keep], y[keep])
        r.update(horizon=h, variant=name); res.append(r)
        tqdm.write(f"  {h:5s} {name:22s} PR-AUC {r['pr_auc']:.3f}  baseline {r['baseline']:.4f}  "
                   f"lift {r['lift']:.1f}x  test positives {r['test_positive_s']:,}")

res = pd.DataFrame(res)
out = res.pivot(index="variant", columns="horizon", values=["pr_auc", "lift"])[
      [(m, h) for m in ["pr_auc", "lift"] for h in CFG["horizons"]]]
print("\n", out.reindex(["All onsets (primary)", "Without blower", "Blower only"]).to_string())
res.to_csv(f"{CFG['outdir']}/blower_performance_check.csv", index=False)

___
**END**